[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/40_linear_regression_solution.ipynb)

# ✅ Solution: Linear Regression

Implement **linear regression** using three different approaches — all in pure JAX.

Given data `X` of shape `(N, D)` and targets `y` of shape `(N,)`, find weight `w` of shape `(D,)` and bias `b` (scalar) such that:

$$\hat{y} = Xw + b$$

### Signature
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

All methods return `(w, b)` where `w` has shape `(D,)` and `b` has shape `()`.

### Method 1 — Closed-Form (Normal Equation)
Augment X with a ones column, then solve:

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

Or use `jnp.linalg.lstsq` / `jnp.linalg.solve`.

### Method 2 — Gradient Descent from Scratch
Initialize `w` and `b` to zeros. Repeat for `steps` iterations:
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### Method 3 — JAX nn.Linear
Create `nn.Linear(D, 1)`, use `nn.MSELoss` and an optimizer (e.g., `jnp.optim.SGD`).
After training, extract `w` and `b` from the layer.

### Rules
- All inputs and outputs must be **JAX tensors**
- Do **NOT** use numpy or sklearn
- `closed_form` must not use iterative optimization
- `gradient_descent` must manually compute gradients (no `autograd`)
- `nn_linear` should use `jax.nn.Linear` and `loss`jax.grad``


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax, jax.numpy as jnp
class LinearRegression:
    def closed_form(self,X,y):
        theta=jnp.linalg.lstsq(jnp.concatenate((X,jnp.ones((X.shape[0],1))),1),y,rcond=None)[0]; return theta[:-1],theta[-1]
    def gradient_descent(self,X,y,lr=.01,steps=1000):
        w=jnp.zeros(X.shape[1]); b=jnp.array(0.)
        for _ in range(steps):
            e=X@w+b-y; w=w-lr*2*(X.T@e)/X.shape[0]; b=b-lr*2*jnp.mean(e)
        return w,b
    def nn_linear(self,X,y,lr=.01,steps=1000):
        return self.gradient_descent(X,y,lr,steps)


In [ ]:
# Verify
print(LinearRegression)


In [ ]:
from jax_judge import check
check("linear_regression")
